[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ersilia-os/ub-cedd-projects-workshop/blob/main/projects/orange/notebooks/orange_essential_proteins_projections.ipynb)

# Map essential proteins with UMAP and t-SNE

**Orange group · Tuberculosis**

Each *M. tuberculosis* protein now has an ESM-C embedding: 960 numbers that describe it. Here we draw all proteins on 2D maps, made with two different methods, where similar proteins end up close together, and highlight the proteins the bacterium needs to survive (the *essential* proteins). This shows whether essential proteins group in particular regions, which helps when choosing drug targets.

## What you will do

- Load the ESM-C embeddings and the list of essential proteins.
- Project the embeddings onto 2D maps with UMAP and with t-SNE.
- Compare where the essential proteins fall on the two maps.

## Setup

Run the cell below first. In Colab it downloads the workshop repository (including the data) and installs the packages this project needs. It takes about a minute. **Don't change it.**

In [ ]:
PROJECT = "orange"
NEEDS_GPU = False
import os, sys, shutil, subprocess
if "google.colab" in sys.modules:
    repo_dir = "/content/ub-cedd-projects-workshop"
    if not os.path.exists(repo_dir):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ersilia-os/ub-cedd-projects-workshop.git", repo_dir], check=True)
    else:
        subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only"], check=True)
    os.chdir(f"{repo_dir}/projects/{PROJECT}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
for _cached in [m for m in sys.modules if m == "scripts" or m.startswith("scripts.")]:
    del sys.modules[_cached]  # forget helper modules imported before the pull above
has_gpu = shutil.which("nvidia-smi") is not None and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
print(f"Python {sys.version.split()[0]} | GPU: {'yes' if has_gpu else 'no'} | Folder: {os.getcwd()}")
if NEEDS_GPU and not has_gpu:
    print("WARNING: this notebook needs a GPU. Go to Runtime > Change runtime type, choose CPU, and run this cell again.")

## 1. Load the embeddings and the essential proteins

We need two files. The first holds the embeddings computed in the previous notebook, with one row per protein. The second is the list of essential proteins, given by their UniProt accession (`UniprotAC`, the unique identifier of each protein).

> **Note:** Both files are in `bigfiles/` at the top of the repository. This folder is not uploaded to GitHub because the embeddings file is large, so for now this notebook runs on a local computer only.

The cell below reads the embeddings and shows the first rows.

In [ ]:
from pathlib import Path
import pandas as pd

BIGFILES = Path("../../bigfiles")
embeddings = pd.read_csv(BIGFILES / "mtb_esmc300m_embeddings.csv")
print(embeddings.shape[0], "proteins,", embeddings.shape[1] - 1, "dimensions")
embeddings.iloc[:5, :6]

Next we read the essential proteins and mark each protein as essential (`True`) or not (`False`). We also check that every essential protein has an embedding.

In [ ]:
essential_acs = set(pd.read_csv(BIGFILES / "essential_uniprot_acs.csv")["uniprot_ac"])
is_essential = embeddings["UniprotAC"].isin(essential_acs)
missing = essential_acs - set(embeddings["UniprotAC"])
print(len(essential_acs), "essential proteins,", is_essential.sum(), "found in the embeddings")
print("Missing:", sorted(missing) if missing else "none")

## 2. Project the embeddings with UMAP

We can't draw 960 dimensions, so we squash them into two. UMAP (*Uniform Manifold Approximation and Projection*) does this while trying to keep proteins that are similar in the embedding close to each other on the map. Distances between far-apart groups on a UMAP map mean little; what matters is which proteins sit together.

We measure similarity with the *cosine* distance, which compares the direction of two embeddings and is common for language-model embeddings. Setting `random_state` makes the map the same every time.

The cell below runs UMAP on the 960 embedding columns and stores the two new coordinates, `x` and `y`. It takes about a minute.

In [ ]:
import umap

dims = [c for c in embeddings.columns if c.startswith("dim_")]
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, metric="cosine", random_state=42)
coords = reducer.fit_transform(embeddings[dims].values)
umap_df = pd.DataFrame({"UniprotAC": embeddings["UniprotAC"], "x": coords[:, 0], "y": coords[:, 1], "essential": is_essential})
umap_df.head()

We save the coordinates, so later notebooks can reuse the map without running UMAP again.

In [ ]:
umap_path = BIGFILES / "mtb_esmc300m_umap.csv"
umap_df.to_csv(umap_path, index=False)
print(umap_path, umap_df.shape)

## 3. Project the embeddings with t-SNE

t-SNE (*t-distributed Stochastic Neighbor Embedding*) is another way to squash the embeddings into two dimensions. Like UMAP, it keeps similar proteins close together, but it focuses even more on the closest neighbours, so it tends to give tighter, more separated groups. Looking at both maps helps: a group that shows up with both methods is more likely to be real.

We use the `openTSNE` package, a fast implementation of t-SNE. The *perplexity* is roughly how many neighbours each protein pays attention to; 30 is a common default.

The cell below runs t-SNE on the same 960 embedding columns, with the same cosine distance and a fixed random seed. It takes less than a minute.

In [ ]:
from openTSNE import TSNE

tsne = TSNE(perplexity=30, metric="cosine", random_state=42, n_jobs=-1)
tsne_coords = tsne.fit(embeddings[dims].values)
tsne_df = pd.DataFrame({"UniprotAC": embeddings["UniprotAC"], "x": tsne_coords[:, 0], "y": tsne_coords[:, 1], "essential": is_essential})
tsne_df.head()

We save these coordinates too, in the same format as the UMAP ones.

In [ ]:
tsne_path = BIGFILES / "mtb_esmc300m_tsne.csv"
tsne_df.to_csv(tsne_path, index=False)
print(tsne_path, tsne_df.shape)

## 4. Highlight the essential proteins

We draw each map with all proteins in light gray and the essential ones in orange on top, so they stand out. The two maps are shown side by side. The axes of UMAP and t-SNE have no meaning of their own; only which proteins are close together matters.

The first cell defines a small function that draws one map. It takes the axis to draw on, so we can reuse it for both maps.

In [ ]:
import stylia

# Format: slide | Style: ersilia — change with stylia.set_format() / stylia.set_style()
stylia.set_format("slide")
stylia.set_style("ersilia")
nc = stylia.NamedColors()

def plot_map(ax, df, title):
    """Draw one 2D map, with essential proteins on top of the rest."""
    rest, ess = df[~df["essential"]], df[df["essential"]]
    ax.scatter(rest["x"], rest["y"], color=nc.get("gray", lighten=0.5), label="Other proteins")
    ax.scatter(ess["x"], ess["y"], color=nc.orange, label="Essential proteins")
    stylia.label(ax, xlabel=f"{title} 1", ylabel=f"{title} 2", title=title)

Now we draw the UMAP map (left) and the t-SNE map (right), and save the figure as `bigfiles/mtb_projections_essential.png`.

In [ ]:
fig, axs = stylia.create_figure(1, 2, width=1.0, height=0.5)  # two square panels
plot_map(axs.next(), umap_df, "UMAP")
ax = axs.next()
plot_map(ax, tsne_df, "t-SNE")
ax.legend()
stylia.save_figure(BIGFILES / "mtb_projections_essential.png")

> **Exercise:** Find a group of essential proteins on the t-SNE map. Filter `tsne_df` for those `x` and `y` values, then check where the same proteins sit in `umap_df`. Are they together on both maps? Look up a few of the accessions on [UniProt](https://www.uniprot.org) and see what they have in common.

## Summary

- We loaded the ESM-C embeddings of the 3,997 *M. tuberculosis* proteins and marked the 337 essential ones.
- We projected the embeddings onto 2D maps with UMAP and with t-SNE, and saved the coordinates in `bigfiles/mtb_esmc300m_umap.csv` and `bigfiles/mtb_esmc300m_tsne.csv`.
- The plots show where the essential proteins fall among the rest of the proteome on both maps.

**Next:** look at what the groups of essential proteins have in common, and add druggability and structure information for the shortlisted targets.